In [ ]:
import numpy as np
import random

class TicTacToeEnv:
    def __init__(self):
        self.board = [0] * 9
        self.current_player = 1

    def reset(self):
        self.board = [0] * 9
        self.current_player = 1
        return tuple(self.board)

    def get_available_actions(self):
        return [i for i, val in enumerate(self.board) if val == 0]

    def step(self, action):
        self.board[action] = self.current_player

        if self.check_win(self.current_player):
            return tuple(self.board), 1.0, True

        if len(self.get_available_actions()) == 0:
            return tuple(self.board), 0.5, True

        self.current_player = -self.current_player
        return tuple(self.board), 0.0, False

    def check_win(self, player):
        win_states = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],
            [0, 3, 6], [1, 4, 7], [2, 5, 8],
            [0, 4, 8], [2, 4, 6]
        ]
        for line in win_states:
            if all(self.board[pos] == player for pos in line):
                return True
        return False

class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0, min_epsilon=0.01, decay_rate=0.0001):
        self.q_table = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.min_epsilon = min_epsilon
        self.decay_rate = decay_rate

    def get_q_values(self, state, actions):
        if state not in self.q_table:
            self.q_table[state] = {action: 0.0 for action in actions}
        return self.q_table[state]

    def choose_action(self, state, actions):
        q_values = self.get_q_values(state, actions)  # register all actions for this state first
        if random.random() < self.epsilon:
            return random.choice(actions)
        max_val = max(q_values.values())
        best_actions = [act for act, val in q_values.items() if val == max_val]
        return random.choice(best_actions)

    def update_q_value(self, state, action, reward, next_state, next_actions, done):
        q_values = self.get_q_values(state, list(self.q_table.get(state, {action: 0.0}).keys()))

        if done:
            target = reward
        else:
            next_q_values = self.get_q_values(next_state, next_actions)
            target = reward + self.gamma * max(next_q_values.values(), default=0.0)

        q_values[action] += self.alpha * (target - q_values[action])

    def decay_epsilon(self, episode):
        self.epsilon = max(self.min_epsilon, np.exp(-self.decay_rate * episode))

In [ ]:
agent = QLearningAgent()
env = TicTacToeEnv()
episodes = 50000

# Training against a random opponent
for episode in range(episodes):
    state = env.reset()
    done = False
    while not done:
        if env.current_player == 1:
            actions = env.get_available_actions()
            action = agent.choose_action(state, actions)
            next_state, reward, done = env.step(action)

            if done:
                agent.update_q_value(state, action, reward, next_state, [], done)
            else:
                opp_actions = env.get_available_actions()
                opp_action = random.choice(opp_actions)
                opp_state, opp_reward, opp_done = env.step(opp_action)

                if opp_done:
                    agent.update_q_value(state, action, -1.0, opp_state, [], opp_done)
                    done = True
                else:
                    next_actions = env.get_available_actions()
                    agent.update_q_value(state, action, 0.0, opp_state, next_actions, False)
                    state = opp_state

    agent.decay_epsilon(episode)

# Testing phase
test_games = 1000
wins, losses, draws = 0, 0, 0
agent.epsilon = 0.0

for _ in range(test_games):
    state = env.reset()
    done = False
    while not done:
        if env.current_player == 1:
            actions = env.get_available_actions()
            action = agent.choose_action(state, actions)
            state, reward, done = env.step(action)
            if done:
                if reward == 1.0:
                    wins += 1
                else:
                    draws += 1
        else:
            opp_actions = env.get_available_actions()
            opp_action = random.choice(opp_actions)
            state, reward, done = env.step(opp_action)
            if done:
                if reward == 1.0:
                    losses += 1
                else:
                    draws += 1

print(f"Results over {test_games} test games:")
print(f"Wins: {wins} ({wins/test_games:.1%})")
print(f"Draws: {draws} ({draws/test_games:.1%})")
print(f"Losses: {losses} ({losses/test_games:.1%})")